In [1]:
import nest_asyncio
nest_asyncio.apply()

In [ ]:
from ib_insync import IB

ib = IB()
ib.connect('127.0.0.1', 4001, clientId=2, readonly=True) #Sometimes there are problems with the cliendID
print(ib.isConnected())

True


Error 1100, reqId -1: Connectivity between IBKR and Trader Workstation has been lost.
Error 1102, reqId -1: Connectivity between IBKR and Trader Workstation has been restored - data maintained. The following farms are connected: usfarm; secdefil. The following farms are not connected: ushmds.
Error 1100, reqId -1: Connectivity between IBKR and Trader Workstation has been lost.
Error 1102, reqId -1: Connectivity between IBKR and Trader Workstation has been restored - data maintained. The following farms are connected: usfarm; secdefil. The following farms are not connected: ushmds.
Error 1100, reqId -1: Connectivity between IBKR and Trader Workstation has been lost.
Error 1102, reqId -1: Connectivity between IBKR and Trader Workstation has been restored - data maintained. The following farms are connected: usfarm; secdefil. The following farms are not connected: ushmds.
Error 1100, reqId -1: Connectivity between IBKR and Trader Workstation has been lost.
Error 1102, reqId -1: Connectivi

In [ ]:
import pandas as pd
from ib_insync import Stock

contract = Stock('SPY', 'SMART', 'USD')

bars = ib.reqHistoricalData(
    contract,
    endDateTime='',
    durationStr='10 Y',
    barSizeSetting='1 day',
    whatToShow='TRADES',
    useRTH=True,
    formatDate=1
)

df = pd.DataFrame(bars)
print(df.shape)
print(df.head())

(2513, 8)
         date    open    high     low   close      volume  average  barCount
0  2016-05-04  204.99  205.85  204.42  205.01  73747383.0  205.045    293803
1  2016-05-05  205.56  205.98  204.47  204.97  58270810.0  205.137    234521
2  2016-05-06  204.06  205.77  203.88  205.72  69470460.0  204.886    283520
3  2016-05-09  205.57  206.40  205.36  205.89  53888569.0  205.918    224980
4  2016-05-10  206.72  208.46  206.64  208.45  55896067.0  207.743    222793


In [14]:
print(df.shape)          # should be ~2520 rows (252 trading days × 10 years)
print(df.dtypes)         # date should be datetime, prices float, volume int
print(df.isnull().sum()) # check for missing values
print(df.head(10))
print(df.tail(10))       # make sure it ends at today / last trading day

(2513, 9)
date         object
open        float64
high        float64
low         float64
close       float64
volume      float64
average     float64
barCount      int64
ticker          str
dtype: object
date        0
open        0
high        0
low         0
close       0
volume      0
average     0
barCount    0
ticker      0
dtype: int64
         date    open    high     low   close      volume  average  barCount  \
0  2016-05-04  204.99  205.85  204.42  205.01  73747383.0  205.045    293803   
1  2016-05-05  205.56  205.98  204.47  204.97  58270810.0  205.137    234521   
2  2016-05-06  204.06  205.77  203.88  205.72  69470460.0  204.886    283520   
3  2016-05-09  205.57  206.40  205.36  205.89  53888569.0  205.918    224980   
4  2016-05-10  206.72  208.46  206.64  208.45  55896067.0  207.743    222793   
5  2016-05-11  207.91  208.54  206.50  206.50  64239291.0  207.358    250942   
6  2016-05-12  207.29  207.49  205.37  206.56  74951582.0  206.411    295395   
7  2016-05-13  20

In [10]:
# After pulling the new 10Y data into df:
import duckdb

# Add ticker column to match the table schema
df['ticker'] = 'SPY'

conn = duckdb.connect('/Users/lucaszelmanovits/Desktop/Quant/Research/Data/quant.db')

# Insert only rows whose date isn't already in the table
conn.execute("""
    INSERT INTO prices_daily
    SELECT date, ticker, open, high, low, close, volume, average, barCount
    FROM df
    WHERE date NOT IN (
        SELECT date FROM prices_daily WHERE ticker = 'SPY'
    )
""")

rows_added = conn.execute("SELECT COUNT(*) FROM prices_daily WHERE ticker = 'SPY'").fetchone()[0]
print(f"Total rows after update: {rows_added}")
conn.close()


Total rows after update: 2513


In [18]:
import duckdb
conn = duckdb.connect('/Users/lucaszelmanovits/Desktop/Quant/Research/Data/quant.db')

result = conn.execute("""
    SELECT date, open, high, low, close, volume 
    FROM prices_daily 
    ORDER BY date ASC 
    LIMIT 10
""").df()

print(result)
conn.close()

        date    open    high     low   close      volume
0 2016-05-04  204.99  205.85  204.42  205.01  73747383.0
1 2016-05-05  205.56  205.98  204.47  204.97  58270810.0
2 2016-05-06  204.06  205.77  203.88  205.72  69470460.0
3 2016-05-09  205.57  206.40  205.36  205.89  53888569.0
4 2016-05-10  206.72  208.46  206.64  208.45  55896067.0
5 2016-05-11  207.91  208.54  206.50  206.50  64239291.0
6 2016-05-12  207.29  207.49  205.37  206.56  74951582.0
7 2016-05-13  206.21  206.86  204.38  204.76  79114901.0
8 2016-05-16  204.96  207.34  204.89  206.78  60847734.0
9 2016-05-17  206.46  206.80  204.23  204.85  93793897.0


In [27]:
conn = duckdb.connect('/Users/lucaszelmanovits/Desktop/Quant/Research/Data/quant.db')
result = conn.execute("SELECT * FROM prices_daily LIMIT 3").df()
print(result.columns.tolist())
conn.close()

['date', 'ticker', 'open', 'high', 'low', 'close', 'volume', 'average', 'barCount', 'log_return', 'rv_21d']


In [19]:
conn = duckdb.connect('/Users/lucaszelmanovits/Desktop/Quant/Research/Data/quant.db')

conn.execute("""
    CREATE OR REPLACE TABLE prices_daily AS
    SELECT *,
        LN(close / LAG(close) OVER (PARTITION BY ticker ORDER BY date)) AS log_return
    FROM prices_daily
""")

conn.close()
print("done")

done


In [30]:
conn = duckdb.connect('/Users/lucaszelmanovits/Desktop/Quant/Research/Data/quant.db')

conn.execute("ALTER TABLE prices_daily ADD COLUMN IF NOT EXISTS rv_5d DOUBLE")
conn.execute("ALTER TABLE prices_daily ADD COLUMN IF NOT EXISTS rv_21d DOUBLE")
conn.execute("ALTER TABLE prices_daily ADD COLUMN IF NOT EXISTS rv_63d DOUBLE")

conn.execute("""
    UPDATE prices_daily
    SET rv_5d  = sub.rv_5d,
        rv_21d = sub.rv_21d,
        rv_63d = sub.rv_63d
    FROM (
        SELECT date, ticker,
            CASE WHEN ROW_NUMBER() OVER (PARTITION BY ticker ORDER BY date) >= 5
                THEN STDDEV_SAMP(log_return) OVER (PARTITION BY ticker ORDER BY date ROWS BETWEEN 4  PRECEDING AND CURRENT ROW) * SQRT(252) END AS rv_5d,
            CASE WHEN ROW_NUMBER() OVER (PARTITION BY ticker ORDER BY date) >= 21
                THEN STDDEV_SAMP(log_return) OVER (PARTITION BY ticker ORDER BY date ROWS BETWEEN 20 PRECEDING AND CURRENT ROW) * SQRT(252) END AS rv_21d,
            CASE WHEN ROW_NUMBER() OVER (PARTITION BY ticker ORDER BY date) >= 63
                THEN STDDEV_SAMP(log_return) OVER (PARTITION BY ticker ORDER BY date ROWS BETWEEN 62 PRECEDING AND CURRENT ROW) * SQRT(252) END AS rv_63d
        FROM prices_daily
    ) sub
    WHERE prices_daily.date = sub.date
    AND prices_daily.ticker = sub.ticker
""")


print(conn.execute("SELECT date, ticker, rv_5d, rv_21d, rv_63d FROM prices_daily WHERE rv_63d IS NOT NULL LIMIT 5").df())
conn.close()


        date ticker     rv_5d    rv_21d    rv_63d
0 2016-08-02    SPY  0.050763  0.078212  0.133050
1 2016-08-03    SPY  0.058307  0.072001  0.132040
2 2016-08-04    SPY  0.058167  0.070351  0.132024
3 2016-08-05    SPY  0.084472  0.073492  0.132714
4 2016-08-08    SPY  0.084174  0.057074  0.132749


In [31]:
conn = duckdb.connect('/Users/lucaszelmanovits/Desktop/Quant/Research/Data/quant.db')

print(conn.execute("""
    SELECT date, ticker, rv_5d, rv_21d, rv_63d 
    FROM prices_daily 
    WHERE ticker = 'SPY'
    ORDER BY date ASC
    LIMIT 70
""").df().to_string())

conn.close()


         date ticker     rv_5d    rv_21d    rv_63d
0  2016-05-04    SPY       NaN       NaN       NaN
1  2016-05-05    SPY       NaN       NaN       NaN
2  2016-05-06    SPY       NaN       NaN       NaN
3  2016-05-09    SPY       NaN       NaN       NaN
4  2016-05-10    SPY  0.090514       NaN       NaN
5  2016-05-11    SPY  0.124138       NaN       NaN
6  2016-05-12    SPY  0.123781       NaN       NaN
7  2016-05-13    SPY  0.140656       NaN       NaN
8  2016-05-16    SPY  0.160791       NaN       NaN
9  2016-05-17    SPY  0.134848       NaN       NaN
10 2016-05-18    SPY  0.125281       NaN       NaN
11 2016-05-19    SPY  0.124652       NaN       NaN
12 2016-05-20    SPY  0.121304       NaN       NaN
13 2016-05-23    SPY  0.090488       NaN       NaN
14 2016-05-24    SPY  0.105504       NaN       NaN
15 2016-05-25    SPY  0.105322       NaN       NaN
16 2016-05-26    SPY  0.090348       NaN       NaN
17 2016-05-27    SPY  0.089613       NaN       NaN
18 2016-05-31    SPY  0.091922 

### VIX

In [23]:
from ib_insync import Index

contract = Index('VIX', 'CBOE', 'USD')

bars = ib.reqHistoricalData(
    contract,
    endDateTime='',
    durationStr='10 Y',
    barSizeSetting='1 day',
    whatToShow='TRADES',
    useRTH=True,
    formatDate=1
)

vix_df = pd.DataFrame(bars)[['date', 'close']]
vix_df.columns = ['date', 'value']
vix_df['series_id'] = 'VIX'
print(vix_df.shape)
print(vix_df.head())


(2513, 3)
         date  value series_id
0  2016-05-04  16.05       VIX
1  2016-05-05  15.91       VIX
2  2016-05-06  14.72       VIX
3  2016-05-09  14.57       VIX
4  2016-05-10  13.63       VIX


In [24]:
conn = duckdb.connect('/Users/lucaszelmanovits/Desktop/Quant/Research/Data/quant.db')

conn.execute("""
    CREATE TABLE IF NOT EXISTS macro_daily (
        date        DATE,
        series_id   VARCHAR,
        value       DOUBLE,
        PRIMARY KEY (date, series_id)
    )
""")

conn.execute("""
    INSERT OR IGNORE INTO macro_daily
    SELECT date, series_id, value FROM vix_df
""")

count = conn.execute("SELECT COUNT(*) FROM macro_daily WHERE series_id = 'VIX'").fetchone()[0]
print(f"VIX rows in DB: {count}")
conn.close()


VIX rows in DB: 2513


### Macro 

In [3]:
from fredapi import Fred
import pandas as pd

FRED_API_KEY = '6c8b2beafa3484a2becf6e8d428f7f76'
fred = Fred(api_key=FRED_API_KEY)

series = {
    '2S10S':     'T10Y2Y',         # Curva de Juros (Já estava aí)
    'HY_OAS':    'BAMLH0A0HYM2',   # High Yield Spread (Medo / Risco de Crédito)
    'ICSA':      'ICSA',           # Pedidos Iniciais de Seguro Desemprego (Macro)
}

start = '2016-05-03'

records = []
for series_id, fred_code in series.items():
    s = fred.get_series(fred_code, observation_start=start)
    s = s.dropna()
    for date, value in s.items():
        records.append({'date': date.date(), 'series_id': series_id, 'value': value})

macro_df = pd.DataFrame(records)
print(macro_df.groupby('series_id').size())


series_id
2S10S     2507
HY_OAS     785
ICSA       522
dtype: int64


note: I have no info in credit spreads for now

In [37]:
conn = duckdb.connect('/Users/lucaszelmanovits/Desktop/Quant/Research/Data/quant.db')

conn.execute("INSERT OR IGNORE INTO macro_daily SELECT date, series_id, value FROM macro_df")

print(conn.execute("SELECT series_id, COUNT(*), MIN(date), MAX(date) FROM macro_daily GROUP BY series_id").df())
conn.close()


  series_id  count_star()  min(date)  max(date)
0       VIX          2513 2016-05-04 2026-05-01
1     2S10S          2500 2016-05-03 2026-05-01


# Features

In [38]:
conn = duckdb.connect('/Users/lucaszelmanovits/Desktop/Quant/Research/Data/quant.db')

for col in ['vol_of_vol', 'vol_term_structure', 'vrp', 'return_autocorr']:
    conn.execute(f"ALTER TABLE prices_daily ADD COLUMN IF NOT EXISTS {col} DOUBLE")

conn.execute("""
    UPDATE prices_daily
    SET vol_of_vol        = sub.vol_of_vol,
        vol_term_structure = sub.vol_term_structure,
        vrp               = sub.vrp,
        return_autocorr   = sub.return_autocorr
    FROM (
        WITH lagged AS (
            SELECT p.date, p.ticker, p.log_return, p.rv_5d, p.rv_21d, p.rv_63d,
                LAG(p.log_return) OVER (PARTITION BY p.ticker ORDER BY p.date) AS lag_return,
                m.value AS vix
            FROM prices_daily p
            LEFT JOIN macro_daily m ON p.date = m.date AND m.series_id = 'VIX'
        )
        SELECT date, ticker,
            CASE WHEN ROW_NUMBER() OVER (PARTITION BY ticker ORDER BY date) >= 21
                THEN STDDEV_SAMP(rv_21d) OVER (PARTITION BY ticker ORDER BY date ROWS BETWEEN 20 PRECEDING AND CURRENT ROW)
            END AS vol_of_vol,

            rv_5d / NULLIF(rv_21d, 0) AS vol_term_structure,

            CASE WHEN vix IS NOT NULL AND rv_21d IS NOT NULL
                THEN (vix / 100.0) - rv_21d
            END AS vrp,

            CASE WHEN ROW_NUMBER() OVER (PARTITION BY ticker ORDER BY date) >= 21
                THEN CORR(log_return, lag_return) OVER (PARTITION BY ticker ORDER BY date ROWS BETWEEN 20 PRECEDING AND CURRENT ROW)
            END AS return_autocorr
        FROM lagged
    ) sub
    WHERE prices_daily.date = sub.date
    AND prices_daily.ticker = sub.ticker
""")

print(conn.execute("""
    SELECT date, vol_of_vol, vol_term_structure, vrp, return_autocorr 
    FROM prices_daily WHERE vol_of_vol IS NOT NULL LIMIT 5
""").df())
conn.close()


        date  vol_of_vol  vol_term_structure       vrp  return_autocorr
0 2016-06-03    0.000988            0.503983  0.034607        -0.381811
1 2016-06-06    0.000699            0.535195  0.035676        -0.394024
2 2016-06-07    0.000589            0.469368  0.039989        -0.387422
3 2016-06-08    0.000510            0.480872  0.040100        -0.385903
4 2016-06-09    0.003211            0.563164  0.053463        -0.414640


In [1]:
import duckdb

conn = duckdb.connect(r'C:\Users\kraus\market_regimes\spy-regime-detection\Data\quant.db') # Ajuste para o seu caminho no Windows

# Criar a coluna
conn.execute("ALTER TABLE prices_daily ADD COLUMN IF NOT EXISTS amihud_liquidity DOUBLE")

# Atualizar a tabela calculando o Amihud diário (multiplicado por 10^6 para a escala ficar legível)
conn.execute("""
    UPDATE prices_daily
    SET amihud_liquidity = sub.amihud
    FROM (
        SELECT date, ticker, 
               (ABS(log_return) / NULLIF(volume, 0)) * 1000000 AS amihud
        FROM prices_daily
    ) sub
    WHERE prices_daily.date = sub.date
    AND prices_daily.ticker = sub.ticker
""")

print(conn.execute("SELECT date, volume, log_return, amihud_liquidity FROM prices_daily ORDER BY date DESC LIMIT 5").df())
conn.close()

        date      volume  log_return  amihud_liquidity
0 2026-05-01  30371385.0    0.008991          0.000296
1 2026-04-30  11935910.0    0.003675          0.000308
2 2026-04-29  28880530.0   -0.000155          0.000005
3 2026-04-28  26375554.0   -0.004878          0.000185
4 2026-04-27  24071338.0    0.001721          0.000072
